# Banten 2024 Electoral Analysis
**Pileg DPRD Provinsi + Pilgub Banten 2024**

Golkar sebagai highlight utama, konteks semua partai.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

df = pd.read_csv('pileg_partai_per_dapil.csv')
df_golkar = pd.read_csv('golkar_per_dapil.csv')
df_pilgub = pd.read_csv('pilgub_per_kab_kota.csv')

# Filter out total row
parties = df[df['partai'] != 'total_suara_sah'].copy()
parties_sorted = parties.sort_values('total', ascending=False)

print('=== RANKING PARTAI PILEG BANTEN 2024 ===')
print(parties_sorted[['partai','total']].to_string(index=False))
print(f'\nTotal suara sah: {df[df["partai"]=="total_suara_sah"]["total"].values[0]:,}')

## 1. Peta Kekuatan Partai — Pileg Banten 2024

In [ ]:
fig, ax = plt.subplots(figsize=(13, 7))

colors = ['#F7C20A' if p == 'Golkar' else '#CCCCCC' for p in parties_sorted['partai']]
bars = ax.barh(parties_sorted['partai'], parties_sorted['total'] / 1000,
               color=colors, edgecolor='white', linewidth=0.5)

for bar, val in zip(bars, parties_sorted['total']):
    ax.text(bar.get_width() + 3, bar.get_y() + bar.get_height()/2,
            f'{val/1000:.0f}K', va='center', fontsize=8.5)

ax.set_xlabel('Total Suara (ribu)', fontsize=11)
ax.set_title('Perolehan Suara Partai — Pileg DPRD Provinsi Banten 2024\n(Golkar = highlight kuning)',
             fontsize=13, fontweight='bold')
ax.axvline(x=0, color='black', linewidth=0.5)

golkar_patch = mpatches.Patch(color='#F7C20A', label='Golkar (Peringkat 1)')
others_patch = mpatches.Patch(color='#CCCCCC', label='Partai lain')
ax.legend(handles=[golkar_patch, others_patch], loc='lower right')

plt.tight_layout()
plt.savefig('01_ranking_partai.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved.')

## 2. Golkar — Perolehan Suara per Dapil

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Left: absolute votes
ax1 = axes[0]
colors_dapil = ['#F7C20A' if v == df_golkar['suara_golkar'].max() else '#E8A800'
                for v in df_golkar['suara_golkar']]
bars1 = ax1.bar(df_golkar['dapil'], df_golkar['suara_golkar'] / 1000,
                color=colors_dapil, edgecolor='white')
ax1.set_xticklabels(df_golkar['dapil'], rotation=45, ha='right', fontsize=9)
ax1.set_ylabel('Suara (ribu)')
ax1.set_title('Golkar: Suara per Dapil', fontweight='bold')
for bar, val in zip(bars1, df_golkar['suara_golkar']):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
             f'{val/1000:.0f}K', ha='center', fontsize=8)

# Right: market share %
ax2 = axes[1]
colors_share = ['#F7C20A' if v == df_golkar['share_pct'].max() else '#E8A800'
                for v in df_golkar['share_pct']]
bars2 = ax2.bar(df_golkar['dapil'], df_golkar['share_pct'],
                color=colors_share, edgecolor='white')
ax2.set_xticklabels(df_golkar['dapil'], rotation=45, ha='right', fontsize=9)
ax2.set_ylabel('Share Suara (%)')
ax2.set_title('Golkar: Market Share per Dapil', fontweight='bold')
ax2.axhline(y=df_golkar['share_pct'].mean(), color='red', linestyle='--', linewidth=1,
            label=f'Rata-rata: {df_golkar["share_pct"].mean():.1f}%')
ax2.legend(fontsize=9)
for bar, val in zip(bars2, df_golkar['share_pct']):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.2,
             f'{val:.1f}%', ha='center', fontsize=8)

plt.suptitle('Golkar — Analisis Per Dapil Banten 2024', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('02_golkar_per_dapil.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved.')

## 3. Top 5 Partai per Dapil — Heatmap

In [ ]:
# Build share matrix
dapil_cols = [f'dapil_{i}' for i in range(1, 13)]
top_parties = ['Golkar', 'Gerindra', 'PDI Perjuangan', 'PKS', 'PKB',
               'Demokrat', 'NasDem', 'PAN']

share_data = {}
total_row = df[df['partai'] == 'total_suara_sah'][dapil_cols].values[0]

for party in top_parties:
    row = df[df['partai'] == party][dapil_cols].values[0]
    share_data[party] = (row / total_row * 100).round(1)

share_df = pd.DataFrame(share_data, index=[f'Dapil {i}' for i in range(1, 13)]).T

fig, ax = plt.subplots(figsize=(14, 6))
im = ax.imshow(share_df.values, cmap='YlOrRd', aspect='auto')

ax.set_xticks(range(12))
ax.set_xticklabels([f'Dapil {i}' for i in range(1, 13)], rotation=45, ha='right')
ax.set_yticks(range(len(top_parties)))
ax.set_yticklabels(top_parties)

for i in range(len(top_parties)):
    for j in range(12):
        val = share_df.values[i, j]
        color = 'white' if val > 18 else 'black'
        # Golkar cells get yellow border
        if top_parties[i] == 'Golkar':
            ax.add_patch(plt.Rectangle((j-0.5, i-0.5), 1, 1,
                         fill=False, edgecolor='#F7C20A', linewidth=2))
        ax.text(j, i, f'{val:.1f}%', ha='center', va='center',
                fontsize=8, color=color, fontweight='bold' if top_parties[i]=='Golkar' else 'normal')

plt.colorbar(im, ax=ax, label='Share Suara (%)')
ax.set_title('Share Suara Top 8 Partai per Dapil — Banten 2024\n(Golkar = border kuning)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('03_heatmap_partai.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved.')

## 4. Pilgub — Peta Kemenangan per Kab/Kota

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))

df_pg = df_pilgub.drop_duplicates(subset='kab_kota').sort_values('andra_pct', ascending=True)

y = range(len(df_pg))
bars_airin = ax.barh(y, df_pg['airin_pct'], color='#3498db', alpha=0.85, label='Airin-Ade')
bars_andra = ax.barh(y, df_pg['andra_pct'], left=df_pg['airin_pct'],
                     color='#e74c3c', alpha=0.85, label='Andra-Dimyati')

ax.axvline(x=50, color='black', linestyle='--', linewidth=1)
ax.set_yticks(y)
ax.set_yticklabels(df_pg['kab_kota'], fontsize=10)
ax.set_xlabel('Persentase Suara (%)')
ax.set_title('Pilgub Banten 2024 — Hasil per Kab/Kota\n(Total: Andra-Dimyati 55.9% vs Airin-Ade 44.1%)',
             fontsize=13, fontweight='bold')
ax.legend(loc='lower right')

for i, (row) in enumerate(df_pg.itertuples()):
    ax.text(row.airin_pct/2, i, f'{row.airin_pct:.1f}%',
            ha='center', va='center', color='white', fontsize=9, fontweight='bold')
    ax.text(row.airin_pct + row.andra_pct/2, i, f'{row.andra_pct:.1f}%',
            ha='center', va='center', color='white', fontsize=9, fontweight='bold')

plt.tight_layout()
plt.savefig('04_pilgub_per_kab_kota.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved.')

## 5. Summary Stats

In [ ]:
total_votes = df[df['partai']=='total_suara_sah']['total'].values[0]
golkar_total = df[df['partai']=='Golkar']['total'].values[0]
golkar_share = golkar_total / total_votes * 100

parties_rank = parties.sort_values('total', ascending=False).reset_index(drop=True)
golkar_rank = parties_rank[parties_rank['partai']=='Golkar'].index[0] + 1

strongest_dapil = df_golkar.loc[df_golkar['share_pct'].idxmax(), 'dapil']
weakest_dapil = df_golkar.loc[df_golkar['share_pct'].idxmin(), 'dapil']

pilgub_total_airin = df_pilgub.drop_duplicates('kab_kota')['airin_ade'].sum()
pilgub_total_andra = df_pilgub.drop_duplicates('kab_kota')['andra_dimyati'].sum()
pilgub_total = pilgub_total_airin + pilgub_total_andra

print('=== SUMMARY BANTEN 2024 ===')
print(f'\n[PILEG]')
print(f'Total suara sah: {total_votes:,}')
print(f'Golkar total: {golkar_total:,} ({golkar_share:.2f}%)')
print(f'Golkar rank nasional di Banten: #{golkar_rank}')
print(f'Golkar strongest dapil: {strongest_dapil} ({df_golkar["share_pct"].max():.1f}%)')
print(f'Golkar weakest dapil: {weakest_dapil} ({df_golkar["share_pct"].min():.1f}%)')
print(f'Rata-rata share per dapil: {df_golkar["share_pct"].mean():.1f}%')

print(f'\n[PILGUB]')
print(f'Airin-Ade: {pilgub_total_airin:,} ({pilgub_total_airin/pilgub_total*100:.1f}%)')
print(f'Andra-Dimyati: {pilgub_total_andra:,} ({pilgub_total_andra/pilgub_total*100:.1f}%)')
print(f'Pemenang: Andra-Dimyati')
print(f'Kantong Airin: Kota Tangerang Selatan, Kota Cilegon')